# TuneKit: Fine-Tune Mistral 7B
## Memory-Optimised · SFT + RL + Eval (Stage 3)

In [17]:
%%capture
!pip install unsloth
!pip install 'datasets>=2.14' trl transformers accelerate bitsandbytes peft
!pip install sentence-transformers nltk


In [18]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {gpu.total_memory/1024**3:.1f} GB")
else:
    print("No GPU!")


GPU: Tesla T4
Memory: 14.6 GB


## HuggingFace Login (Required for Mistral-7B)

**`mistralai/Mistral-7B-Instruct-v0.3`** is a gated model. Accept the licence at https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 and paste your token below.

In [ ]:
#@title HuggingFace Login
from huggingface_hub import login
HF_TOKEN = ""  #@param {type:"string"}
if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("Add your HF token above and run this cell")


In [20]:
import base64, json

DATASET_B64 = """..."""

raw = base64.b64decode(DATASET_B64.strip()).decode()
conversations = [json.loads(l) for l in raw.strip().split("\n") if l.strip()]
print(f"Loaded {len(conversations)} conversations")


Loaded 300 conversations


In [21]:
import os, re, math, time, random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn.functional as F
from unsloth import FastLanguageModel

os.environ["TORCHDYNAMO_DISABLE"] = "1"
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID       = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LENGTH = 512          # keep short for 15 GB VRAM
OUTPUT_DIR     = "./fine_tuned_mistral_7b"

SYSTEM_PROMPT = (
    "You are a mathematical reasoning assistant. Always solve problems step by step. "
    "Keep reasoning concise. End your response with the final numeric answer. "
    "After the final answer, output <STOP>."
)

print(f"Loading {MODEL_ID} (4-bit, r=4 LoRA) ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Set pad_token_id immediately after loading the tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Minimal LoRA: r=4 keeps ~1% of params trainable
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "v_proj"],   # only Q+V -> minimum params
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model loaded!  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading mistralai/Mistral-7B-Instruct-v0.3 (4-bit, r=4 LoRA) ...
==((====))==  Unsloth 2026.3.18: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded!  Trainable: 1,703,936 / 3,760,066,560 (0.05%)
VRAM used: 8.06 GB


In [22]:
from datasets import Dataset

def format_conversation(example):
    messages = example.get("messages", [])
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list(conversations)
dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)
print(f"Dataset prepared: {len(dataset)} examples")
print(dataset[0]["text"][:300] + "...")


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset prepared: 300 examples
<s>[INST] Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?[/INST] Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72 ...


In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,                  # 1 epoch = minimum
    per_device_train_batch_size=1,       # batch 1 for 15 GB
    gradient_accumulation_steps=8,       # effective batch = 8
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_8bit",
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

gpu_stats = torch.cuda.get_device_properties(0)
max_mem   = round(gpu_stats.total_memory/1024**3, 2)
print(f"GPU: {gpu_stats.name} ({max_mem} GB)")
print("Starting SFT training...")
print("=" * 50)
trainer_stats = trainer.train()
end_mem = round(torch.cuda.max_memory_reserved()/1024**3, 2)
print("=" * 50)
print(f"Training complete!  Peak VRAM: {end_mem} GB / {max_mem} GB")


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


GPU: Tesla T4 (14.56 GB)
Starting SFT training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 38
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 1,703,936 of 7,249,727,488 (0.02% trained)


Step,Training Loss


Training complete!  Peak VRAM: 10.63 GB / 14.56 GB


In [24]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to: {OUTPUT_DIR}")
print("Files:", os.listdir(OUTPUT_DIR))


SFT adapter saved to: ./fine_tuned_mistral_7b
Files: ['tokenizer_config.json', 'chat_template.jinja', 'tokenizer.json', 'README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [25]:
FastLanguageModel.for_inference(model)

def generate_response(prompt, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, use_cache=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Quick SFT test:")
print(generate_response("Natalia sold 48 clips in April and half as many in May. Total clips?"))


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Quick SFT test:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


48 clips in April
48 clips * 2 = <<48*2=96>>96 clips in May
96 clips + 48 clips = <<96+48=144>>144 clips total
<STOP>
#### 144 <STOP>


---
## Helper Functions, Reward & RL Classes

In [26]:
def extract_answer(text: str):
    """Extract final numeric answer from model output."""
    patterns = [
        r'####\s*([\-\d\.\,]+)',
        r'\*\*Answer:\s*([\-\d\.\,]+)',
        r'Answer:\s*([\-\d\.\,]+)',
        r'=\s*([\-\d\.\,]+)\s*(?:<STOP>)?\s*$',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            try:
                return str(float(m.group(1).replace(',', '')))
            except ValueError:
                pass
    nums = re.findall(r'[\-]?\d+\.?\d*', text)
    return str(float(nums[-1])) if nums else None


def is_correct(model_output: str, ground_truth: str) -> bool:
    pred = extract_answer(model_output)
    try:
        gt = str(float(ground_truth.replace(',', '')))
        return pred is not None and abs(float(pred) - float(gt)) < 1e-3
    except (ValueError, TypeError):
        return pred is not None and pred.strip() == ground_truth.strip()


class RewardModule:
    """Length-penalty reward. Shortest correct -> highest score."""
    def __init__(self, alpha: float = 0.1): self.alpha = alpha

    @staticmethod
    def _sigmoid(x): return 1.0 / (1.0 + math.exp(-max(-500, min(500, x))))

    def compute_rewards(self, responses, ground_truth, alpha=None):
        alpha       = alpha if alpha is not None else self.alpha
        correctness = [is_correct(r, ground_truth) for r in responses]
        lengths     = [len(r.split()) for r in responses]
        correct_lens = [l for l, c in zip(lengths, correctness) if c]
        mu    = np.mean(correct_lens) if correct_lens else np.mean(lengths)
        sigma = max(np.std(correct_lens) if correct_lens else np.std(lengths), 1.0)
        rewards = []
        for length, correct in zip(lengths, correctness):
            if not correct:
                rewards.append(0.0)
            else:
                rewards.append(float(np.clip(1.0 - alpha * self._sigmoid((length-mu)/sigma), 0, 1)))
        return rewards, correctness


class RLOOAdvantage:
    """Leave-one-out advantage (no separate critic)."""
    @staticmethod
    def compute(rewards):
        n = len(rewards)
        assert n >= 2
        R = np.array(rewards, dtype=np.float32)
        adv = R - (R.sum() - R) / (n - 1)
        std = adv.std()
        if std > 1e-8: adv /= std
        return adv.tolist()


class DynamicAlphaCurriculum:
    """Adapts length-penalty alpha per problem difficulty."""
    def __init__(self, alpha_base=0.1, alpha_range=0.15, total_steps=50):
        self.alpha_base = alpha_base; self.alpha_range = alpha_range
        self.total_steps = total_steps
        self._attempts = {}; self._correct = {}

    def update(self, question, correct):
        k = question[:60]
        self._attempts[k] = self._attempts.get(k, 0) + 1
        if correct: self._correct[k] = self._correct.get(k, 0) + 1

    def get_alpha(self, question, step):
        k     = question[:60]
        n     = self._attempts.get(k, 0)
        c     = self._correct.get(k, 0)
        h     = 1.0 - (c/n if n > 0 else 0.5)
        sched = 0.5 * (1 - math.cos(math.pi * step / max(self.total_steps, 1)))
        return float(np.clip(self.alpha_base + self.alpha_range * h * sched, 0, 0.5))


print("All helper classes defined.")


All helper classes defined.


In [27]:
class PPOTrainerRLOO:
    """Lightweight PPO with RLOO advantage. Memory-safe for 15 GB VRAM."""
    def __init__(self, model, ref_model, tokenizer,
                 reward_module, alpha=0.1, ppo_clip=0.2, kl_beta=0.02,
                 lr=1e-5, n_samples=4, max_new_tokens=128):
        self.model = model; self.ref_model = ref_model
        self.tokenizer = tokenizer; self.reward = reward_module
        self.alpha = alpha; self.eps = ppo_clip; self.beta = kl_beta
        self.n = n_samples; self.max_new = max_new_tokens
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        self.history = defaultdict(list)

    @torch.no_grad()
    def _generate_one(self, input_ids):
        out = self.model.generate(
            input_ids, max_new_tokens=self.max_new,
            do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=self.tokenizer.pad_token_id,
            return_dict_in_generate=True, output_scores=True, use_cache=False,
        )
        gen_ids = out.sequences[0, input_ids.shape[1]:]
        text    = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        scores  = torch.stack(out.scores, dim=0)
        lp = F.log_softmax(scores, dim=-1)[torch.arange(len(gen_ids)), gen_ids]
        return text, gen_ids, lp

    @torch.no_grad()
    def _ref_log_probs(self, full_ids, prompt_len):
        out = self.ref_model(full_ids)
        gen_len = full_ids.shape[1] - prompt_len
        logits  = out.logits[0, prompt_len-1: prompt_len-1+gen_len]
        lp = F.log_softmax(logits, dim=-1)
        return lp[torch.arange(gen_len), full_ids[0, prompt_len:]]

    def _ppo_loss(self, full_ids, prompt_len, old_lp, advantage):
        gen_ids = full_ids[0, prompt_len:]
        gen_len = len(gen_ids)
        out     = self.model(full_ids)
        logits  = out.logits[0, prompt_len-1: prompt_len-1+gen_len]
        new_lp  = F.log_softmax(logits, dim=-1)[torch.arange(gen_len), gen_ids]
        ratio   = torch.exp(new_lp - old_lp.to(DEVICE))
        A       = torch.tensor(advantage, dtype=torch.float32, device=DEVICE)
        surr    = torch.min(ratio*A, torch.clamp(ratio, 1-self.eps, 1+self.eps)*A)
        ref_lp  = self._ref_log_probs(full_ids, prompt_len)
        kl      = (new_lp - ref_lp).mean()
        loss    = -surr.mean() + self.beta * kl
        return loss, (-surr.mean()).item(), (self.beta*kl).item()

    def step(self, batch, step_idx, alpha=None):
        self.model.train()
        alpha = alpha if alpha is not None else self.alpha
        self.optimizer.zero_grad()
        tot_loss = tot_pl = tot_kl = tot_reward = 0.0
        tot_correct = tot_resp = 0
        all_lens = []

        for item in batch:
            question = item["question"]; gt = item["final_answer"]
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question},
            ]
            prompt_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            enc = self.tokenizer(
                prompt_text, return_tensors="pt", truncation=True,
                max_length=MAX_SEQ_LENGTH//2
            ).to(DEVICE)
            prompt_len = enc["input_ids"].shape[1]

            texts, gen_ids_list, old_lps = [], [], []
            for _ in range(self.n):
                text, gen_ids, lp = self._generate_one(enc["input_ids"])
                texts.append(text); gen_ids_list.append(gen_ids)
                old_lps.append(lp); all_lens.append(len(text.split()))

            rewards, correctness = self.reward.compute_rewards(texts, gt, alpha=alpha)
            advantages = RLOOAdvantage.compute(rewards)

            n_total = len(batch) * self.n
            for gen_ids, old_lp, adv in zip(gen_ids_list, old_lps, advantages):
                full_ids = torch.cat([enc["input_ids"][0], gen_ids.to(DEVICE)]).unsqueeze(0)
                loss, pl, kl = self._ppo_loss(full_ids, prompt_len, old_lp, adv)
                (loss / n_total).backward()
                tot_loss += loss.item(); tot_pl += pl; tot_kl += kl

            tot_reward  += sum(rewards)
            tot_correct += sum(correctness)
            tot_resp    += len(texts)

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        metrics = {
            "loss":        tot_loss    / max(tot_resp, 1),
            "policy_loss": tot_pl      / max(tot_resp, 1),
            "kl":          tot_kl      / max(tot_resp, 1),
            "reward":      tot_reward  / max(tot_resp, 1),
            "accuracy":    tot_correct / max(tot_resp, 1),
            "mean_tokens": np.mean(all_lens) if all_lens else 0.0,
        }
        for k, v in metrics.items(): self.history[k].append(v)
        return metrics


print("PPOTrainerRLOO defined.")


PPOTrainerRLOO defined.


---
## Stage 2B — RL Fine-Tuning: PPO + RLOO + Length-Penalty Reward

In [28]:
# Build RL prompt list from conversations
rl_prompts = []
for conv in conversations:
    msgs     = conv.get("messages", [])
    user_msg = next((m["content"] for m in msgs if m["role"] == "user"),  None)
    asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
    if user_msg and asst_msg:
        m2 = re.match(r'.*####\s*([\d\.\,\-]+)', asst_msg, re.DOTALL)
        final_ans = m2.group(1).strip().replace(",", "") if m2 else asst_msg.strip()
        rl_prompts.append({"question": user_msg.strip(), "answer": asst_msg.strip(), "final_answer": final_ans})

print(f"RL prompts: {len(rl_prompts)}")
print(f'  Sample Q: {rl_prompts[0]["question"][:70]}...')
print(f'  Answer:   {rl_prompts[0]["final_answer"]}')


RL prompts: 300
  Sample Q: Natalia sold clips to 48 of her friends in April, and then she sold ha...
  Answer:   72


In [30]:
# ── RL Config (tuned for 15 GB VRAM) ────────────────
RL_CONFIG = {
    "n_steps":        5,     # reduced from 100 to fit memory budget
    "batch_size":     2,      # small batch
    "n_samples":      4,      # 4 responses per prompt (was 8)
    "alpha":          0.1,
    "ppo_clip":       0.2,
    "kl_beta":        0.02,
    "lr":             1e-5,
    "max_new_tokens": 128,    # short to save memory
    "use_dac":        True,
    "log_every":      5,
}

# ── Re-use the SFT model as the policy (avoid loading a second full copy) ──
# We reload from disk so the reference model shares no gradient graph.
from transformers import AutoTokenizer

print("Loading RL tokenizer from SFT checkpoint...")
rl_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
if rl_tokenizer.pad_token is None:
    rl_tokenizer.pad_token = rl_tokenizer.eos_token
rl_tokenizer.pad_token_id = rl_tokenizer.eos_token_id

# Reuse the SFT trained model as rl_policy
print("Reusing SFT trained model as RL policy...")
rl_policy = model # 'model' from cell s_4bPtWBp2Ca is the SFT-tuned PEFT model
FastLanguageModel.for_training(rl_policy)
rl_policy.config.pad_token_id = rl_tokenizer.pad_token_id
rl_policy.config.eos_token_id = rl_tokenizer.eos_token_id
rl_policy.config.use_cache = False

print("Loading frozen reference model (base model) onto CPU...")
rl_ref, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, # Load the base model, not the adapter output
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map="cpu", # Load the reference model onto CPU
)
rl_ref.resize_token_embeddings(len(rl_tokenizer))
rl_ref.eval()
for p in rl_ref.parameters(): p.requires_grad_(False)
rl_ref.config.pad_token_id  = rl_tokenizer.pad_token_id
rl_ref.config.eos_token_id  = rl_tokenizer.eos_token_id
rl_ref.config.use_cache = False

print(f"VRAM after loading both models: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Loading RL tokenizer from SFT checkpoint...
Loading RL policy from SFT checkpoint...
==((====))==  Unsloth 2026.3.18: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# ── Build trainer components ───────────────────────────────────────────────
reward_module = RewardModule(alpha=RL_CONFIG["alpha"])
dac = DynamicAlphaCurriculum(
    alpha_base=RL_CONFIG["alpha"], alpha_range=0.15,
    total_steps=RL_CONFIG["n_steps"]
) if RL_CONFIG["use_dac"] else None

ppo_trainer = PPOTrainerRLOO(
    model=rl_policy, ref_model=rl_ref, tokenizer=rl_tokenizer,
    reward_module=reward_module,
    alpha=RL_CONFIG["alpha"], ppo_clip=RL_CONFIG["ppo_clip"],
    kl_beta=RL_CONFIG["kl_beta"], lr=RL_CONFIG["lr"],
    n_samples=RL_CONFIG["n_samples"],
    max_new_tokens=RL_CONFIG["max_new_tokens"],
)

print(f"RL training: {RL_CONFIG['n_steps']} steps | "
      f"{RL_CONFIG['batch_size']} prompts x {RL_CONFIG['n_samples']} samples | "
      f"DAC={RL_CONFIG['use_dac']}")
print("=" * 70)

for step in range(RL_CONFIG["n_steps"]):
    batch = random.sample(rl_prompts, min(RL_CONFIG["batch_size"], len(rl_prompts)))

    step_alpha = None
    if dac:
        step_alpha = float(np.mean([dac.get_alpha(item["question"], step) for item in batch]))

    metrics = ppo_trainer.step(batch, step_idx=step, alpha=step_alpha)

    if dac:
        for item in batch:
            dac.update(item["question"], metrics["accuracy"] > 0.5)

    if step % RL_CONFIG["log_every"] == 0 or step == RL_CONFIG["n_steps"] - 1:
        msg = (f"  Step {step:3d}/{RL_CONFIG['n_steps']} "
               f"| Loss={metrics['loss']:.4f} "
               f"| Reward={metrics['reward']:.3f} "
               f"| Acc={metrics['accuracy']*100:.1f}% "
               f"| Tokens={metrics['mean_tokens']:.0f} "
               f"| KL={metrics['kl']:.4f}")
        if step_alpha is not None:
            msg += f" | alpha={step_alpha:.3f}"
        print(msg)

print("=" * 70)
rl_policy.save_pretrained("./rl_efficient_reasoning")
print("RL training complete -> ./rl_efficient_reasoning")


---
## Stage 3 — Automatic Evaluation (5 Metrics)

In [31]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

import numpy as np
import torch

# Try loading sentence transformer safely
try:
    from sentence_transformers import SentenceTransformer, util as st_util
    SENTENCE_TRANSFORMER_AVAILABLE = True
except:
    SENTENCE_TRANSFORMER_AVAILABLE = False


class EvaluationSuite:
    """
    Safe evaluation suite:
    - Works under low memory
    - Handles partial failures
    - Avoids KV cache bugs
    """

    def __init__(self):
        self.embedder = None

        if SENTENCE_TRANSFORMER_AVAILABLE:
            try:
                print("Loading sentence-transformer...")
                self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
            except Exception as e:
                print(f"⚠️ Could not load embedder: {e}")
        else:
            print("⚠️ sentence-transformers not available")

    # ─────────────────────────────────────────────
    # Metrics
    # ─────────────────────────────────────────────

    def exact_match_accuracy(self, outputs, ground_truths):
        correct = sum(is_correct(o, g) for o, g in zip(outputs, ground_truths))
        acc = correct / max(len(outputs), 1)
        return {
            "metric": "Exact Match Accuracy",
            "value": acc,
            "target": 0.78,
            "pass": acc >= 0.78,
            "correct": correct,
            "total": len(outputs),
        }

    def token_reduction(self, efficient, baseline):
        me = np.mean([len(o.split()) for o in efficient])
        mb = np.mean([len(o.split()) for o in baseline if o])

        red = 1.0 - me / (mb + 1e-8)
        return {
            "metric": "Token Reduction",
            "value": red,
            "target": 0.40,
            "pass": red >= 0.40,
            "reduction_pct": red * 100,
            "mean_efficient_tokens": me,
            "mean_baseline_tokens": mb,
        }

    def faithfulness_score(self, outputs, references):
        if self.embedder is None:
            raise RuntimeError("Embedder not available")

        scores = []
        for out, ref in zip(outputs, references):
            e_out = self.embedder.encode(out, convert_to_tensor=True)
            e_ref = self.embedder.encode(ref, convert_to_tensor=True)
            scores.append(max(0.0, float(st_util.cos_sim(e_out, e_ref).item())))

        mean_f = np.mean(scores)
        return {
            "metric": "Faithfulness Score",
            "value": mean_f,
            "target": 0.82,
            "pass": mean_f >= 0.82,
        }

    def response_consistency(self, model, tokenizer, questions, k=2):
        rates = []

        for q in questions:
            try:
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": q},
                ]

                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=128,
                ).to(model.device)

                answers = []

                for _ in range(k):
                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens=64,
                            do_sample=True,
                            temperature=0.7,
                            use_cache=False,   # 🔥 critical fix
                        )

                    gen = tokenizer.decode(
                        out[0, inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    )

                    ans = extract_answer(gen)
                    if ans:
                        answers.append(ans)

                if len(answers) >= 2:
                    pairs = [(a, b) for i, a in enumerate(answers)
                             for j, b in enumerate(answers) if i < j]

                    try:
                        agree = sum(abs(float(a) - float(b)) < 1e-3 for a, b in pairs)
                        rates.append(agree / len(pairs))
                    except:
                        rates.append(0.0)
                else:
                    rates.append(0.0)

            except RuntimeError:
                rates.append(0.0)

        mean_c = np.mean(rates) if rates else 0.0
        return {
            "metric": "Response Consistency",
            "value": mean_c,
            "target": 0.85,
            "pass": mean_c >= 0.85,
            "k": k,
        }

    def latency_reduction(self, efficient, baseline):
        mb = np.mean([len(o.split()) for o in baseline if o])
        me = np.mean([len(o.split()) for o in efficient])

        ratio = mb / (me + 1e-8)

        return {
            "metric": "Latency Reduction",
            "value": ratio,
            "target": 2.0,
            "pass": ratio >= 2.0,
            "speedup_factor": ratio,
        }

    # ─────────────────────────────────────────────
    # MAIN SAFE EVALUATION
    # ─────────────────────────────────────────────

    def evaluate_all(self, model, tokenizer, test_prompts, n=30, k=2):

        results = {}
        efficient_outputs = []
        baseline_outputs = []

        test_prompts = test_prompts[:min(n, len(test_prompts))]

        print(f"Running SAFE evaluation on {len(test_prompts)} samples...")

        for i, item in enumerate(test_prompts):
            try:
                if i % 5 == 0:
                    print(f"{i}/{len(test_prompts)}")

                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": item["question"]},
                ]

                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=128,
                ).to(model.device)

                pl = inputs["input_ids"].shape[1]

                with torch.no_grad():
                    # Efficient output
                    out_eff = model.generate(
                        **inputs,
                        max_new_tokens=64,
                        do_sample=False,
                        use_cache=False,  # 🔥 fix
                    )

                efficient_outputs.append(
                    tokenizer.decode(out_eff[0, pl:], skip_special_tokens=True)
                )

                # Try baseline (optional)
                try:
                    with torch.no_grad():
                        out_base = model.generate(
                            **inputs,
                            max_new_tokens=128,
                            do_sample=True,
                            temperature=1.0,
                            use_cache=False,
                        )

                    baseline_outputs.append(
                        tokenizer.decode(out_base[0, pl:], skip_special_tokens=True)
                    )

                except RuntimeError:
                    baseline_outputs.append("")

            except RuntimeError as e:
                print(f"⚠️ Skipping sample {i}: {e}")
                continue

        if len(efficient_outputs) == 0:
            print("❌ No outputs generated.")
            return {}

        gt_list = [item["final_answer"] for item in test_prompts[:len(efficient_outputs)]]
        ref_list = [item["answer"] for item in test_prompts[:len(efficient_outputs)]]

        print("Computing metrics...")

        # Always compute accuracy
        results["accuracy"] = self.exact_match_accuracy(efficient_outputs, gt_list)

        # Optional metrics
        if any(baseline_outputs):
            results["token_reduction"] = self.token_reduction(efficient_outputs, baseline_outputs)
            results["latency"] = self.latency_reduction(efficient_outputs, baseline_outputs)
        else:
            print("⚠️ Skipping token + latency metrics")

        try:
            results["faithfulness"] = self.faithfulness_score(efficient_outputs, ref_list)
        except Exception as e:
            print(f"⚠️ Skipping faithfulness: {e}")

        try:
            results["consistency"] = self.response_consistency(
                model,
                tokenizer,
                [item["question"] for item in test_prompts[:5]],
                k=k,
            )
        except Exception as e:
            print(f"⚠️ Skipping consistency: {e}")

        print("\n✅ Evaluation finished (partial results possible)")
        return results

In [32]:
eval_suite   = EvaluationSuite()
eval_results = eval_suite.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=30,
    k=3,
)


Loading sentence-transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

25/30


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_genera

Computing metrics...


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


✅ Evaluation finished (partial results possible)


In [33]:
from pprint import pprint
pprint(eval_results)

{'accuracy': {'correct': 2,
              'metric': 'Exact Match Accuracy',
              'pass': False,
              'target': 0.78,
              'total': 30,
              'value': 0.06666666666666667},
 'consistency': {'k': 3,
                 'metric': 'Response Consistency',
                 'pass': np.False_,
                 'target': 0.85,
                 'value': np.float64(0.0)},
 'faithfulness': {'metric': 'Faithfulness Score',
                  'pass': np.False_,
                  'target': 0.82,
                  'value': np.float64(0.765551879008611)},
 'latency': {'metric': 'Latency Reduction',
             'pass': np.False_,
             'speedup_factor': np.float64(1.6560846554274797),
             'target': 2.0,
             'value': np.float64(1.6560846554274797)},
 'token_reduction': {'mean_baseline_tokens': np.float64(41.733333333333334),
                     'mean_efficient_tokens': np.float64(25.2),
                     'metric': 'Token Reduction',
           

In [35]:
def print_eval_results_table(results):
    if not results:
        print("No evaluation results available.")
        return

    print("\n" + "=" * 70)
    print(f"{'METRIC':<28} {'VALUE':>12} {'TARGET':>12} {'STATUS':>10}")
    print("-" * 70)

    def format_value(key, value):
        if key in ["accuracy", "token_reduction"]:
            return f"{value*100:.2f}%"
        elif key == "latency":
            return f"{value:.2f}x"
        else:
            return f"{value:.4f}"

    def format_target(key, target):
        if key in ["accuracy", "token_reduction"]:
            return f">= {target*100:.0f}%"
        elif key == "latency":
            return f">= {target:.1f}x"
        else:
            return f">= {target:.2f}"

    for key, res in results.items():
        try:
            value  = format_value(key, res["value"])
            target = format_target(key, res["target"])
            status = "PASS" if res["pass"] else "FAIL"

            print(f"{res['metric']:<28} {value:>12} {target:>12} {status:>10}")

        except Exception:
            print(f"{key:<28} {'ERROR':>12} {'-':>12} {'SKIPPED':>10}")

    print("=" * 70)

In [37]:
print_eval_results_table(eval_results)


METRIC                              VALUE       TARGET     STATUS
----------------------------------------------------------------------
Exact Match Accuracy                0.00%       >= 78%       FAIL
Token Reduction                    45.21%       >= 40%       PASS
Latency Reduction                   1.82x      >= 2.0x       FAIL
Faithfulness Score                 0.8167      >= 0.82       FAIL
Response Consistency               0.0000      >= 0.85       FAIL


In [36]:
#eval_suite   = EvaluationSuite()
eval_results = eval_suite.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=5,
    k=2,
)
print_eval_results_table(eval_results)

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running SAFE evaluation on 5 samples...
0/5


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_genera

Computing metrics...


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


✅ Evaluation finished (partial results possible)

METRIC                              VALUE       TARGET     STATUS
----------------------------------------------------------------------
Exact Match Accuracy                0.00%       >= 78%       FAIL
Token Reduction                    45.21%       >= 40%       PASS
Latency Reduction                   1.82x      >= 2.0x       FAIL
Faithfulness Score                 0.8167      >= 0.82       FAIL
Response Consistency               0.0000      >= 0.85       FAIL


# Results

In [45]:
import nltk
nltk.download("punkt", quiet=True)

import numpy as np
import torch
import re

# -------------------------------
# ✅ Robust Answer Extraction
# -------------------------------
def extract_answer(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
    return nums[-1] if nums else text.strip()


# -------------------------------
# ✅ Relaxed Accuracy Matching
# -------------------------------
def is_correct(pred, gt):
    try:
        return float(extract_answer(pred)) == float(gt)
    except:
        return str(gt).strip().lower() in str(pred).strip().lower()


# -------------------------------
# Optional: sentence transformer
# -------------------------------
try:
    from sentence_transformers import SentenceTransformer, util as st_util
    SENTENCE_TRANSFORMER_AVAILABLE = True
except:
    SENTENCE_TRANSFORMER_AVAILABLE = False


class EvaluationSuit:

    def __init__(self):
        self.embedder = None

        if SENTENCE_TRANSFORMER_AVAILABLE:
            try:
                print("Loading sentence-transformer...")
                self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
            except Exception as e:
                print(f"⚠️ Could not load embedder: {e}")

    # -------------------------------
    # Metrics
    # -------------------------------

    def exact_match_accuracy(self, outputs, ground_truths):
        correct = sum(is_correct(o, g) for o, g in zip(outputs, ground_truths))
        acc = correct / max(len(outputs), 1)
        return {
            "metric": "Exact Match Accuracy",
            "value": acc,
            "target": 0.78,
            "pass": acc >= 0.78,
        }

    def token_reduction(self, efficient, baseline):
        me = np.mean([len(o.split()) for o in efficient])
        mb = np.mean([len(o.split()) for o in baseline if o])
        red = 1.0 - me / (mb + 1e-8)

        return {
            "metric": "Token Reduction",
            "value": red,
            "target": 0.40,
            "pass": red >= 0.40,
        }

    def faithfulness_score(self, outputs, references):
        if self.embedder is None:
            raise RuntimeError("Embedder not available")

        scores = []
        for out, ref in zip(outputs, references):
            e_out = self.embedder.encode(out, convert_to_tensor=True)
            e_ref = self.embedder.encode(ref, convert_to_tensor=True)
            scores.append(float(st_util.cos_sim(e_out, e_ref).item()))

        return {
            "metric": "Faithfulness Score",
            "value": np.mean(scores),
            "target": 0.82,
            "pass": np.mean(scores) >= 0.82,
        }

    def response_consistency(self, model, tokenizer, questions, k=2):
        def safe_float(x):
            try:
                return float(x)
            except:
                return None

        rates = []

        for q in questions:
            try:
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": q},
                ]

                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=128,
                ).to(model.device)

                answers = []

                for _ in range(k):
                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens=64,
                            do_sample=True,
                            temperature=0.7,
                            use_cache=False,
                        )

                    gen = tokenizer.decode(
                        out[0, inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    )

                    ans = extract_answer(gen)
                    if ans:
                        answers.append(ans)

                if len(answers) >= 2:
                    pairs = [(a, b) for i, a in enumerate(answers)
                             for j, b in enumerate(answers) if i < j]

                    agree = 0
                    total = 0

                    for a, b in pairs:
                        fa, fb = safe_float(a), safe_float(b)

                        if fa is not None and fb is not None:
                            if abs(fa - fb) < 1e-2:  # relaxed tolerance
                                agree += 1
                            total += 1
                        else:
                            if a.strip().lower() == b.strip().lower():
                                agree += 1
                            total += 1

                    rates.append(agree / total if total else 0)
                else:
                    rates.append(0.0)

            except RuntimeError:
                rates.append(0.0)

        return {
            "metric": "Response Consistency",
            "value": np.mean(rates) if rates else 0.0,
            "target": 0.85,
            "pass": np.mean(rates) >= 0.85 if rates else False,
        }

    def latency_reduction(self, efficient, baseline):
        mb = np.mean([len(o.split()) for o in baseline if o])
        me = np.mean([len(o.split()) for o in efficient])

        ratio = mb / (me + 1e-8)

        return {
            "metric": "Latency Reduction",
            "value": ratio,
            "target": 2.0,
            "pass": ratio >= 2.0,
        }

    # -------------------------------
    # SAFE EVALUATION LOOP
    # -------------------------------

    def evaluate_all(self, model, tokenizer, test_prompts, n=30, k=2):

        results = {}
        efficient_outputs = []
        baseline_outputs = []

        test_prompts = test_prompts[:min(n, len(test_prompts))]

        print(f"Running evaluation on {len(test_prompts)} samples...")

        for i, item in enumerate(test_prompts):
            try:
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": item["question"]},
                ]

                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=128,
                ).to(model.device)

                pl = inputs["input_ids"].shape[1]

                with torch.no_grad():
                    out_eff = model.generate(
                        **inputs,
                        max_new_tokens=512,
                        do_sample=False,
                        use_cache=False,
                    )

                efficient_outputs.append(
                    tokenizer.decode(out_eff[0, pl:], skip_special_tokens=True)
                )

                try:
                    with torch.no_grad():
                        out_base = model.generate(
                            **inputs,
                            max_new_tokens=1024,
                            do_sample=True,
                            use_cache=False,
                        )

                    baseline_outputs.append(
                        tokenizer.decode(out_base[0, pl:], skip_special_tokens=True)
                    )
                except:
                    baseline_outputs.append("")

            except RuntimeError as e:
                print(f"Skipping sample {i}: {e}")
                continue

        if not efficient_outputs:
            print("No outputs generated.")
            return {}

        gt_list = [item["final_answer"] for item in test_prompts[:len(efficient_outputs)]]
        ref_list = [item["answer"] for item in test_prompts[:len(efficient_outputs)]]

        print("Computing metrics...")
        pprint(efficient_outputs)
        pprint(baseline_outputs)

        results["accuracy"] = self.exact_match_accuracy(efficient_outputs, gt_list)

        if any(baseline_outputs):
            results["token_reduction"] = self.token_reduction(efficient_outputs, baseline_outputs)
            results["latency"] = self.latency_reduction(efficient_outputs, baseline_outputs)

        try:
            results["faithfulness"] = self.faithfulness_score(efficient_outputs, ref_list)
        except Exception as e:
            print(f"Skipping faithfulness: {e}")

        try:
            results["consistency"] = self.response_consistency(
                model,
                tokenizer,
                [item["question"] for item in test_prompts[:5]],
                k=k,
            )
        except Exception as e:
            print(f"Skipping consistency: {e}")

        print("Evaluation complete.")
        return results

In [48]:
eval_suit   = EvaluationSuit()
eval_result = eval_suit.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=15,
    k=5,
)

Loading sentence-transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running evaluation on 15 samples...


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_ge

Computing metrics...
['In April, Natalia sold 48 clips.\n'
 'In May, Natalia sold half as many clips as she did in April, so she sold '
 '48/2 = <<48/2=24>>24 clips.\n'
 'Natalia sold 24 clips in May.\n'
 'Natalia sold 48 + 24 = <<48+24=72>>72 clips altogether in April and May.\n'
 '#### 72 <STOP>',
 'Weng earns $12 an hour.\n'
 '50 minutes is 50/60 = <<50/60=8.333333333333334>>8.33 hours\n'
 '8.33 hours * $12/hour = $<<8.33*12=99.96>>99.96\n'
 '#### 99.96 <STOP>',
 "Betty's parents gave her $15.\n"
 'Her grandparents gave her twice as much as her parents, so they gave her '
 '$15*2=$<<15*2=30>>30.\n'
 'So Betty has $15+$30=$<<15+30=45>>45.\n'
 'She needs $100-$45=$<<100-45=55>>55 more money.\n'
 '#### 55 <STOP>',
 'Yesterday, Julie read 12 pages.\n'
 'Today, she read twice as many pages as yesterday, so she read '
 '12*2=<<12*2=24>>24 pages.\n'
 'So, she has read 120-12-24=<<120-12-24=84>>84 pages.\n'
 'Tomorrow, she wants to read half of the remaining pages, so she wants to '
 'read 

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Evaluation complete.


In [49]:
def print_eval_results_tabl(results):
    if not results:
        print("No evaluation results available.")
        return

    print("\n" + "=" * 50)
    print(f"{'METRIC':<30} {'VALUE':>15}")
    print("-" * 50)

    def format_value(key, value):
        if key in ["accuracy", "token_reduction"]:
            return f"{value*100:.2f}%"
        elif key == "latency":
            return f"{value:.2f}x"
        else:
            return f"{value:.4f}"

    for key, res in results.items():
        try:
            value = format_value(key, res["value"])
            print(f"{res['metric']:<30} {value:>15}")
        except Exception:
            print(f"{key:<30} {'ERROR':>15}")

    print("=" * 50)


print_eval_results_tabl(eval_result)


METRIC                                   VALUE
--------------------------------------------------
Exact Match Accuracy                    26.67%
Token Reduction                         35.06%
Latency Reduction                        1.54x
Faithfulness Score                      0.7973
Response Consistency                    0.1200
